# CHORDATA — Split-download-compatible upload

The verified 64/64 source is available as the original ZIP, a short-name identical ZIP, or 6 smaller parts. Upload **all parts at once** in the first Colab code cell. It reassembles and verifies the identical SHA-256 before processing.

The QA branch guard and manual `PUSH` confirmation remain unchanged. Scientific sign-off is not claimed.


In [ ]:
from pathlib import Path
import hashlib, json, os, zipfile, subprocess, shutil, sys, tempfile

REPO = "rameshgascngl-create/Chordata"
BRANCH = "native/qa-chordata-illustration-coverage"
EXPECTED_HEAD = "5371d7ecfc8a03955ba86100578dad6538c5fd26"
EXPECTED_FILE = "CHORDATA_64of64_Hardened_GitHub_Upload_QA_Source_20261107.zip"
EXPECTED_SHA256 = "50918a3544e46b70de3174092942ec78085c2eb9bb00a7756897847a0e9e07c2"

# Upload one complete ZIP (original filename or short-name alias), OR all 6 small parts together.
# The SHA-256 gate below rejects incomplete, corrupted or different source packages.
from google.colab import files
PART_NAMES = ['CHORDATA_64_QA_20261107.part01', 'CHORDATA_64_QA_20261107.part02', 'CHORDATA_64_QA_20261107.part03', 'CHORDATA_64_QA_20261107.part04', 'CHORDATA_64_QA_20261107.part05', 'CHORDATA_64_QA_20261107.part06']
ALIASED_FILE = 'CHORDATA_64_QA_Source.zip'
uploaded = files.upload()
if EXPECTED_FILE in uploaded:
    zip_bytes = uploaded[EXPECTED_FILE]
elif ALIASED_FILE in uploaded:
    zip_bytes = uploaded[ALIASED_FILE]
else:
    missing = [name for name in PART_NAMES if name not in uploaded]
    assert not missing, "STOP: missing split parts: " + ", ".join(missing)
    zip_bytes = b"".join(uploaded[name] for name in PART_NAMES)
    print("Reassembled", len(PART_NAMES), "parts into", len(zip_bytes), "bytes")
actual_sha256 = hashlib.sha256(zip_bytes).hexdigest()
print("Uploaded ZIP SHA-256:", actual_sha256)
assert actual_sha256 == EXPECTED_SHA256, "STOP: archive checksum does not match the validated source"

work = Path(tempfile.mkdtemp(prefix="chordata-qa-"))
archive = work / EXPECTED_FILE
archive.write_bytes(zip_bytes)
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None, "STOP: corrupt ZIP member"
    members = z.infolist()
    assert len(members) >= 150, "STOP: incomplete project ZIP"
    for info in members:
        pp = Path(info.filename)
        assert not pp.is_absolute() and ".." not in pp.parts and pp.parts[0] == "Chordata",             f"STOP: unsafe member {info.filename}"
        assert ((info.external_attr >> 16) & 0o170000) != 0o120000, "STOP: symlink in ZIP"
    z.extractall(work / "extracted")
source = work / "extracted" / "Chordata"
assert (source / ".github/workflows/android-qa.yml").is_file()
assets = source / "app/src/main/assets"
registry = json.loads((assets / "figure_slots.json").read_text())
manifest = json.loads((assets / "pencil_plates/manifest.json").read_text())
assert len(registry["slots"]) == 64 and len(manifest["figures"]) == 68
assert len(list(assets.rglob("*.png"))) == 74
assert len(list((source / "docs/original_plate_svg").glob("*.svg"))) == 32
print("PASS: expected ZIP, 64 diagram slots, 68 variants, 74 PNGs, 32 SVG masters")

In [ ]:
# Run all Python-only verification gates before changing the remote.
checks = [
    ["scripts/audit_content.py"],
    ["scripts/audit_figure_placement.py"],
    ["scripts/audit_pencil_coverage.py"],
    ["scripts/audit_coverage_report.py"],
    ["scripts/audit_android_qa_contract.py"],
    ["scripts/audit_preview_bitmap_budget.py"],
    ["scripts/audit_built_apk_assets.py", "--self-test"],
    ["scripts/test_apk_source_fixture.py"],
    ["scripts/write_figure_review_register.py", "--check"],
    ["scripts/test_review_signoff_guards.py"],
    ["scripts/test_sdk37_platform.py"],
    ["scripts/test_git_sync_whitespace.py"],
    ["scripts/check_textbook_plates.py"],
    ["scripts/validate_credited_sources.py"],
]
for args in checks:
    subprocess.run([sys.executable, *args], cwd=source, check=True)
print("PASS: all Python QA checks. Kotlin core and Android compilation are CI gates; not claimed here.")

## Push to the isolated QA branch

Run the next cell only after source checks pass. Your token is requested using a private password-style field. The next cell will display the changes and ask you to type `PUSH`. If the branch is no longer at `5371d7e`, **stop and reconcile**; this notebook will not overwrite it.

In [ ]:
from getpass import getpass
import base64

assert shutil.which("git"), "Git must be installed in this Colab runtime"
assert shutil.which("rsync"), "rsync must be installed in this Colab runtime"
token = getpass("GitHub token (hidden; use a short-lived repo-scoped token): ").strip()
assert token, "No token supplied"
header = "AUTHORIZATION: basic " + base64.b64encode(("x-access-token:" + token).encode()).decode()
authenv = os.environ.copy()
authenv.update({
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": header,
    "GIT_TERMINAL_PROMPT": "0"
})
# Delete the original cleartext token variable immediately; the auth environment
# is kept only for clone/push and removed before finishing.
del token, header

def git(*args, cwd=None, auth=False, capture=False):
    r = subprocess.run(["git", *args], cwd=cwd,
        env=authenv if auth else None, text=True, check=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.PIPE if capture else None)
    return r.stdout.strip() if capture else ""

try:
    checkout = work / "repo"
    git("clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
        f"https://github.com/{REPO}.git", str(checkout), auth=True)
    remote_head = git("rev-parse", "HEAD", cwd=checkout, capture=True)
    new_head = remote_head
    assert remote_head == EXPECTED_HEAD, (
        "STOP: QA branch changed; expected " + EXPECTED_HEAD + ", found " + remote_head)

    subprocess.run(["rsync", "-a", "--delete", "--exclude=.git/",
                    "--exclude=__pycache__/", "--exclude=*.py[cod]",
                    "--exclude=.pytest_cache/",
                    str(source) + "/", str(checkout) + "/"], check=True)
    git("add", "-A", cwd=checkout)
    git("diff", "--cached", "--check", cwd=checkout)
    # Preflight tools generate Python caches. No such files may enter the commit.
    staged_paths = subprocess.run(
        ["git", "diff", "--cached", "--name-only", "-z"],
        cwd=checkout, check=True, stdout=subprocess.PIPE).stdout.decode().split("\0")
    assert not any(
        "/__pycache__/" in "/" + p or p.endswith((".pyc", ".pyo"))
        for p in staged_paths if p
    ), "STOP: generated Python bytecode was staged"
    # The exact number of source illustrations must be present in the staging tree.
    assert len(list((checkout / "app/src/main/assets").rglob("*.png"))) == 74, (
        "STOP: not all 74 PNG assets reached the checkout")
    print("Changes in the QA branch only:")
    git("diff", "--cached", "--stat", cwd=checkout)
    changed = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=checkout).returncode != 0
    if not changed:
        print("Already synchronised: nothing to push.")
    else:
        approval = input("Type PUSH to commit and upload to the isolated QA branch: ").strip()
        assert approval == "PUSH", "Upload cancelled; no changes pushed"
        git("-c", "user.name=rameshgascngl-create",
            "-c", "user.email=242819382+rameshgascngl-create@users.noreply.github.com",
            "commit", "-m", "qa(chordata): synchronise complete 64-plate provisional source and Android CI gates",
            cwd=checkout)
        # Non-force push; any intervening remote commit causes a rejection.
        git("push", "origin", f"HEAD:refs/heads/{BRANCH}", cwd=checkout, auth=True)
        new_head = git("rev-parse", "HEAD", cwd=checkout, capture=True)
        print("PUSHED QA BRANCH:", BRANCH, "NEW HEAD:", new_head)
    print("CI: https://github.com/rameshgascngl-create/Chordata/actions")
    print("No production merge or release was performed. Review the new CI build and screenshots.")
finally:
    # The token and auth header are not saved to the repository or notebook file.
    authenv.pop("GIT_CONFIG_VALUE_0", None)
    authenv.clear()
    del authenv


## Optional: verify CI and download its debug APK

Only use this **after a successful QA-branch push**. The cell checks GitHub Actions for the exact pushed commit and downloads an APK **only if the workflow concludes successfully**. It then runs the repository's own 80-asset APK integrity audit before offering the APK for download. If CI fails, it prints the run link rather than claiming an APK exists. Your GitHub token is prompted again (hidden) and is not saved in the notebook.

This is a **debug QA APK**, not a production release. Scientific and Tamil sign-off and physical-device review remain pending.

In [ ]:
import requests, time, io, zipfile
from getpass import getpass
from google.colab import files as colab_files

assert "new_head" in globals(), "Run the sync/push cell before checking CI"
api_token = getpass("GitHub token to read CI artifacts (hidden): ").strip()
assert api_token, "Token required for downloading workflow artifacts"
session = requests.Session()
session.headers.update({
    "Authorization": "Bearer " + api_token,
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
})
del api_token

try:
    api = f"https://api.github.com/repos/{REPO}"
    run = None
    for attempt in range(60):
        response = session.get(api + "/actions/runs", params={
            "branch": BRANCH, "head_sha": new_head, "per_page": 20
        }, timeout=30)
        response.raise_for_status()
        candidates = [r for r in response.json().get("workflow_runs", [])
            if r.get("head_sha") == new_head and r.get("name") == "Chordata Native QA Debug APK"]
        if candidates:
            run = max(candidates, key=lambda r: r.get("id", 0))
            if run.get("status") == "completed": break
        if attempt % 6 == 0:
            print("CI verification pending for commit", new_head[:12])
        time.sleep(20)
    assert run is not None, "No GitHub Actions run found for the pushed QA commit"
    print("Workflow:", run["html_url"], "Status:", run["status"], "Conclusion:", run.get("conclusion"))
    assert run["status"] == "completed", "Workflow has not completed; use the link above later"
    assert run.get("conclusion") == "success", "STOP: CI failed. Inspect workflow logs; do not use an unverified APK"

    response = session.get(api + f"/actions/runs/{run['id']}/artifacts", timeout=30)
    response.raise_for_status()
    artifacts = [a for a in response.json().get("artifacts", [])
                 if a.get("name") == "Chordata_Native_QA_Debug" and not a.get("expired")]
    assert artifacts, "CI succeeded, but its debug APK artifact is absent or expired"
    artifact = artifacts[0]
    download = session.get(api + f"/actions/artifacts/{artifact['id']}/zip", timeout=90)
    download.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(download.content)) as z:
        assert z.testzip() is None, "Damaged GitHub Actions artifact ZIP"
        entries = [n for n in z.namelist() if n.endswith("app-debug.apk")]
        assert len(entries) == 1, "Expected exactly one debug APK inside the artifact"
        apk_path = work / "CHORDATA_64of64_GitHub_QA_Debug.apk"
        apk_path.write_bytes(z.read(entries[0]))
    subprocess.run([sys.executable, "scripts/audit_built_apk_assets.py", str(apk_path)],
                   cwd=source, check=True)
    print("PASS: APK asset audit, SHA-256", hashlib.sha256(apk_path.read_bytes()).hexdigest())
    colab_files.download(str(apk_path))
finally:
    session.headers.pop("Authorization", None)
    session.close()
